In [1]:
from dotenv import load_dotenv
load_dotenv()
import os
from supabase import create_client, Client
supabase_url = os.environ.get("SUPABASE_URL")
supabase_api_key = os.environ.get("SUPBASE_KEY")

supabase: Client = create_client(supabase_url, supabase_api_key)

In [2]:
from openai import OpenAI
client = OpenAI(api_key = os.getenv("OPENAI_API_KEY"))

In [3]:
def report_research(date, next_date):
    result = supabase.table('curation_selected_items')\
        .select('event_id, output,sources, news_date','topic, created_at')\
    .gte('created_at', date)\
    .lt('created_at', next_date)\
    .eq('topic', 'Report')\
        .order('created_at', desc=False)\
        .execute()
    
    return result.data

In [21]:
report_deep_dive = report_research('2026-03-02', '2026-03-03')

In [22]:
report_deep_dive

[{'event_id': '1909_2026-02-26',
  'output': "Thomson Reuters Institute published the 2026 'AI in Professional Services' report (Feb 26, 2026). Key findings: generative AI use nearly doubled year-over-year and is now weekly for most organizations; only ~18% of organizations actively measure AI ROI; many measure internal metrics (cost/efficiency) rather than revenue/client impact. The report includes a concrete example (Thomson Reuters CoCounsel Legal combined with Supio) showing measurable workflow impact in personal injury litigation (structured medical records, faster case preparation, scalable caseloads and improved client communications), indicating enterprise/industry-case evidence of measurable AI business impact in professional services.",
  'sources': [{'url': 'https://legal.thomsonreuters.com/blog/highlights-from-the-2026-ai-in-professional-services-report-and-what-it-means-for-legal-teams-tri/',
    'name': 'Thomson Reuters'}],
  'news_date': '2026-02-26',
  'topic': 'Report'

In [23]:
def openai_research_workflow(research_input):
    for i in research_input:
        response = client.responses.create(
            model = "gpt-5-nano",
            tools = [{
                "type": "web_search"
            }],
            include = ["web_search_call.action.sources"],
            input = f"""
            You are a senior research analyst for Krux.

            TASK:
              Create a deep, fact-checked brief for ONE AI report/paper/benchmark release.
            
            GOAL:
              Extract the highest-value findings and implementation guidance so a later 100-word summary can capture most practical signal.
            
            OUTPUT FORMAT:
              - 12 to 16 bullet points.
              - Each bullet must include inline citations:
                [Source: <publisher>, <url>]
              - No text before or after bullets.
            
            MANDATORY STRUCTURE:
              1) REPORT SNAPSHOT
              - What was published, by whom, and when.
              - Scope: geography, sectors, sample size, time window, method type.
            
              2) CORE FINDINGS (MOST IMPORTANT)
              - 4 to 6 bullets with the strongest quantified findings.
              - Include exact numbers, not vague language.
            
              3) WHAT THIS ACTUALLY MEANS
              - 2 to 3 bullets translating findings into plain English implications for real teams.
            
              4) IMPLEMENTATION PLAYBOOK
              - 3 to 4 bullets: what organizations should do in the next quarter.
              - Include sequencing (e.g., data readiness -> pilot -> measurement -> rollout).
            
              5) METRICS TO TRACK
              - 1 or 2 bullets on KPIs teams should monitor to apply the report in practice.
            
              6) LIMITATIONS / CAVEATS
              - 1 or 2 bullets on methodology limits, sample bias, missing data, or uncertainty.
            
              7) DECISION TAKEAWAY
              - 1 bullet: “If a team only does one thing based on this report, it should be X.”
            
              STRICT RULES:
              - No hype, no opinion, no generic AI commentary.
              - Every factual claim must be source-backed.
              - If a key detail is missing, write: "Not disclosed in cited sources."
              - Prefer primary sources (original report/paper) over secondary articles.
              - If secondary coverage conflicts with primary report, prioritize primary and note conflict.
            
              QUALITY CHECK:
              - Must contain quantified findings.
              - Must contain actionable implementation steps.
              - Must clearly separate findings from interpretation.
              - Must include limitations.
            
              Report/event to research:
              Report summary: {i['output']}
              Source: {i['sources']}
            """,
        )

        output = response.output_text
        print(output)

        final_dictionary = {
            'event_id': i['event_id'],
            'news_date': i['news_date'],
            'output': output,
            'model_provider': 'openai',
            'topic': i['topic']
        }
        print(final_dictionary)

        save_research(final_dictionary)

        print("✅ Done")

In [24]:
openai_research_workflow(report_deep_dive)

- REPORT SNAPSHOT: 2026 AI in Professional Services Report (special report) by Thomson Reuters Institute; released February 9, 2026; surveys GenAI and agentic AI adoption across the legal, tax & accounting, risk/fraud, and government sectors. [Source: Thomson Reuters Institute, https://www.thomsonreuters.com/en-us/posts/technology/ai-in-professional-services-report-2026/]

- REPORT SNAPSHOT: The study draws on perspectives from more than 1,500 professionals across 27 countries, covering legal, tax & accounting, corporate functions, and government agencies. [Source: Thomson Reuters Institute, https://www.thomsonreuters.com/en-us/posts/technology/ai-in-professional-services-report-2026/]

- CORE FINDINGS: GenAI organization-wide adoption nearly doubled to 40% in 2026 (vs 22% in 2025), with most respondents already experimenting or using GenAI. [Source: Thomson Reuters Institute, https://www.thomsonreuters.com/en-us/posts/technology/ai-in-professional-services-report-2026/]

- CORE FINDIN

In [11]:
def save_research(research_json):
    supabase.table('research_assistant').insert({
        'event_id': research_json['event_id'],
        'model_provider': research_json['model_provider'],
        'news_date': research_json['news_date'],
        'output': research_json['output'],
        'topic': research_json['topic']
    }).execute()